In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import torch
from pytorch_implementation.fbbev.config import debug_forward_config
from pytorch_implementation.fbbev.model import FBBEVLite

cfg = debug_forward_config(max_num=48, depth_bins=6, bev_h=20, bev_w=20, bev_z=3)
model = FBBEVLite(cfg).eval()

batch_size = 2
height, width = 96, 160
img = torch.randn(batch_size, cfg.num_cams, 3, height, width)

def _build_dummy_img_metas(batch_size):
    metas = []
    for batch_idx in range(batch_size):
        curr_to_prev = torch.eye(4, dtype=torch.float32)
        curr_to_prev[0, 3] = 0.4 * (batch_idx + 1)
        curr_to_prev[1, 3] = -0.2 * (batch_idx + 1)
        metas.append({
            "sample_idx": f"sample_{batch_idx}",
            "sequence_group_idx": batch_idx,
            "start_of_sequence": True,
            "curr_to_prev_ego_rt": curr_to_prev,
        })
    return metas
img_metas = _build_dummy_img_metas(batch_size)


print(f"Model: {type(model).__name__}")
print(f"Config: {cfg.name}")
print(f"Input: {tuple(img.shape)}")

In [ ]:
from typing import Any

def _first_tensor(value: Any):
    """Extract the first tensor from a nested structure."""
    import torch
    if torch.is_tensor(value):
        return value
    if isinstance(value, (tuple, list)):
        for item in value:
            t = _first_tensor(item)
            if t is not None:
                return t
    if isinstance(value, dict):
        for item in value.values():
            t = _first_tensor(item)
            if t is not None:
                return t
    return None

def _iter_tensors(value: Any):
    """Iterate over all tensors in a nested structure."""
    import torch
    if torch.is_tensor(value):
        yield value
    elif isinstance(value, (tuple, list)):
        for item in value:
            yield from _iter_tensors(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from _iter_tensors(item)

def _register_hook(module, name: str, capture: dict, handles: list) -> None:
    """Register a forward hook that stores output in capture[name]."""
    def _hook(_module, _inputs, output):
        capture[name] = output
    handles.append(module.register_forward_hook(_hook))

def _print_shape(label: str, value) -> None:
    """Print shape of first tensor in value."""
    t = _first_tensor(value)
    if t is not None:
        print(f"  {label}: {tuple(t.shape)}")
    else:
        print(f"  {label}: <no tensor>")

def _check_finite(capture: dict, outputs: dict) -> None:
    """Assert all captured intermediates and outputs are finite."""
    import torch
    for name, value in capture.items():
        for t in _iter_tensors(value):
            assert torch.isfinite(t).all(), f"Non-finite in {name}"
    for name, value in outputs.items():
        if value is None:
            continue
        for t in _iter_tensors(value):
            assert torch.isfinite(t).all(), f"Non-finite in output {name}"
    print("All intermediate and final tensors are finite.")

# FB-BEV Paper-to-Code Study Guide

This note maps FB-BEV paper symbols/equations to the pure-PyTorch forward implementation in this repository.

Primary references:
- Paper: `/media/farrosalferro/College/study/3d_perception/papers/FB-BEV.pdf`
- Reference implementation: `/media/farrosalferro/College/study/3d_perception/repos/FB-BEV/`
- Pure-PyTorch implementation: `pytorch_implementation/fbbev/`
- Intermediate tensor tests: `tests/fbbev/test_intermediate_tensors.py`

## 1) Canonical study setup (fixed debug run)

Use one setup so equation-to-tensor mapping stays stable across sections.

- Config:
  - `debug_forward_config(max_num=48, depth_bins=6, bev_h=20, bev_w=20, bev_z=3)`
- Input image:
  - `img`: `[B, Ncam, C, H, W] = [2, 6, 3, 96, 160]`
- Metadata (`img_metas`) per sample:
  - `sequence_group_idx`
  - `start_of_sequence`
  - `curr_to_prev_ego_rt`: `4 x 4` ego transform matrix

Core dimensions under this setup:
- `embed_dims = 128`
- `num_classes = 10`
- `depth_bins = 6`
- `bev = [Hbev, Wbev, Zbev] = [20, 20, 3]`
- `max_num = 48`

Expected model outputs:
- `all_cls_scores`: `[L, B, Q, num_classes] = [1, 2, 48, 10]`
- `all_bbox_preds`: `[L, B, Q, code_size] = [1, 2, 48, 9]`

These are verified in `tests/fbbev/test_intermediate_tensors.py`.

## 2) Symbol dictionary (paper -> code tensors)

- `F` (image-view feature tensor) -> `camera_feat` / `context` in `FBBEVLite.forward`
- `D` (depth distribution) -> `depth` in `FBBEVDepthNetLite.forward`
- `B` (lifted BEV volume) -> `bev_volume` in `ForwardProjectionLite.forward`
- `B'` (depth-aware refined BEV) -> `bev_refined` in `BackwardProjectionLite.forward`
- `M` (temporal memory/history) -> `self.history_bev` in `TemporalFusionLite`
- `t_f` (temporal offset embedding) -> `time_map` in `TemporalFusionLite.forward`
- `Q_{x,y}` (BEV query locations) -> dense BEV grid positions in `FBBEVDetectionHeadLite._meshgrid`
- `N_c` (camera count) -> `cfg.num_cams`
- `N_ref` (depth reference bins) -> `cfg.depth_bins`
- `w_c` (depth consistency weight) -> `depth_weight` in `DepthAwareAttentionLite.forward`

Equation IDs below are stable and use `E<section>.<index>`.

---

## Chunk 0 - End-to-end forward contract

### Goal
Bind FB-BEV high-level 3D detection pipeline to concrete module calls.

### Paper concept/equation
FB-BEV combines forward projection, backward depth-aware refinement, and temporal BEV fusion before detection.

### Explicit equations
`(E0.1)` Pipeline:

$$
F = \mathrm{ImageEncoder}(I),\;
D = \mathrm{DepthNet}(F),\;
B = \mathrm{ForwardProj}(F, D),\;
B' = \mathrm{BackwardProj}(B, F, D)
$$

`(E0.2)` Temporal fusion and detection:

$$
\tilde{B} = \mathrm{TemporalFuse}(B', M, t_f),\quad
\hat{Y} = \mathrm{DetHead}(\mathrm{BEVEncoder}(\tilde{B}))
$$

### Symbol table (E0.*)
- `I`: multi-camera image input
- `M`: queued historical BEV features
- `\hat{Y}`: class and box predictions

### Code mapping
- `FBBEVLite.forward` in `pytorch_implementation/fbbev/model.py`
- `FBBEVDepthNetLite.forward` in `pytorch_implementation/fbbev/depth_net.py`
- `ForwardProjectionLite.forward` in `pytorch_implementation/fbbev/forward_projection.py`
- `BackwardProjectionLite.forward` in `pytorch_implementation/fbbev/backward_projection.py`
- `TemporalFusionLite.forward` in `pytorch_implementation/fbbev/temporal_fusion.py`
- `FBBEVDetectionHeadLite.forward` in `pytorch_implementation/fbbev/detection_head.py`

### One sanity check
`tests/fbbev/test_intermediate_tensors.py` checks final output shapes for the debug config.

---

In [ ]:
# Chunk 0: End-to-end forward
# Source: pytorch_implementation/fbbev/model.py - FBBEVLite.forward

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)

print("=== FB-BEV output shapes ===")
for key, val in outputs.items():
    if val is not None and torch.is_tensor(val):
        print(f"  {key}: {tuple(val.shape)}")

assert outputs["all_cls_scores"].shape[0] == 1  # single head layer
print("\nShape assertions passed.")

## Chunk 1 - Camera projection and depth weighting

### Goal
Connect depth probabilities to BEV volume construction.

### Paper concept/equation
Use depth logits to weight per-camera context, then aggregate across cameras.

### Explicit equations
`(E1.1)` Depth probabilities:

$$
D = \mathrm{softmax}(W_d * F)
$$

`(E1.2)` Lift and aggregate:

$$
B = \frac{1}{N_c}\sum_{i=1}^{N_c} \left(F_i \odot D_i\right)
$$

### Symbol table (E1.*)
- `W_d`: depth prediction conv
- `\odot`: element-wise depth weighting

### Code mapping
- `FBBEVDepthNetLite` (`context_proj`, `depth_logits`) in `pytorch_implementation/fbbev/depth_net.py`
- `ForwardProjectionLite.forward` in `pytorch_implementation/fbbev/forward_projection.py`

### Tensor shape notes
- `context`: `[B, Ncam, C, Hf, Wf]`
- `depth`: `[B, Ncam, D, Hf, Wf]`
- `bev_volume`: `[B, C, Hbev, Wbev, Zbev]`

### One sanity check
Tests assert `depth_net.context_proj`, `depth_net.depth_logits`, and `forward_projection` output shapes.

---

In [ ]:
# Chunk 1: Camera projection and depth weighting
# Source: pytorch_implementation/fbbev/depth_net.py
# Source: pytorch_implementation/fbbev/forward_projection.py

capture, handles = {}, []
_register_hook(model.depth_net.context_proj, "depth_net.context_proj", capture, handles)
_register_hook(model.depth_net.depth_logits, "depth_net.depth_logits", capture, handles)
_register_hook(model.forward_projection, "forward_projection", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

print("=== Depth net and forward projection ===")
_print_shape("depth_net.context_proj", capture["depth_net.context_proj"])
_print_shape("depth_net.depth_logits", capture["depth_net.depth_logits"])
_print_shape("forward_projection (bev_volume)", capture["forward_projection"])
print(f"\n  context output: {tuple(outputs['context'].shape)}")
print(f"  depth output:   {tuple(outputs['depth'].shape)}")
print(f"  bev_volume:     {tuple(outputs['bev_volume'].shape)}")

## Chunk 2 - Depth-aware backward projection

### Goal
Map depth-consistency weighting to BEV refinement.

### Paper concept/equation
Depth-aware attention reweights context-to-BEV aggregation using consistency term `w_c`.

### Explicit equations
`(E2.1)` Depth-consistency weight:

$$
w_c = \sigma\left(\langle q, k \rangle\right)\cdot d_{prior}
$$

`(E2.2)` Depth-aware update:

$$
B' = \mathrm{Conv}\left(q + w_c \cdot v\right)
$$

### Symbol table (E2.*)
- `q, k, v`: projected BEV/context features
- `d_{prior}`: reduced depth confidence map

### Code mapping
- `DepthAwareAttentionLite.forward` in `pytorch_implementation/fbbev/depth_aware_attention.py`
- `BackwardProjectionLite.forward` in `pytorch_implementation/fbbev/backward_projection.py`

### Tensor shape notes
- `bev_2d`: `[B, C, Hbev, Wbev]`
- `depth_weight`: `[B, 1, Hbev, Wbev]`
- `bev_refined`: `[B, C, Hbev, Wbev]`

### One sanity check
Tests assert `backward.depth_attention` and `backward.post` intermediate shapes.

---

In [ ]:
# Chunk 2: Depth-aware backward projection
# Source: pytorch_implementation/fbbev/backward_projection.py
# Source: pytorch_implementation/fbbev/depth_aware_attention.py

capture, handles = {}, []
_register_hook(model.backward_projection.depth_attention,
               "backward.depth_attention", capture, handles)
_register_hook(model.backward_projection.post_conv,
               "backward.post", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

print("=== Backward projection shapes ===")
_print_shape("depth_attention", capture["backward.depth_attention"])
_print_shape("post_conv (bev_refined)", capture["backward.post"])
print(f"\n  bev_refined: {tuple(outputs['bev_refined'].shape)}")

## Chunk 3 - Temporal fusion with ego-motion alignment

### Goal
Tie history queue update to temporal transform input.

### Paper concept/equation
Current BEV is fused with aligned history using relative ego-motion and temporal offset embeddings.

### Explicit equations
`(E3.1)` History alignment:

$$
M_t^{align} = \mathcal{W}(M_{t-1}, T_{t\rightarrow t-1})
$$

`(E3.2)` Time-aware fusion:

$$
\tilde{B}_t = \phi\left(\left[B_t,\; M_t^{align},\; t_f\right]\right)
$$

### Symbol table (E3.*)
- `\mathcal{W}`: warping/alignment operator
- `T_{t\rightarrow t-1}`: `curr_to_prev_ego_rt`
- `\phi`: small conv fusion network

### Code mapping
- `TemporalFusionLite._warp_single` and `TemporalFusionLite.forward` in `pytorch_implementation/fbbev/temporal_fusion.py`

### Tensor shape notes
- history queue: `[B, T, C, Hbev, Wbev]`
- fused output: `[B, C, Hbev, Wbev]`

### One sanity check
Tests assert `temporal.time_conv` and `temporal.cat_conv` outputs.

---

In [ ]:
# Chunk 3: Temporal fusion with ego-motion alignment
# Source: pytorch_implementation/fbbev/temporal_fusion.py

capture, handles = {}, []
_register_hook(model.temporal_fusion.time_conv, "temporal.time_conv", capture, handles)
_register_hook(model.temporal_fusion.cat_conv, "temporal.cat_conv", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

print("=== Temporal fusion shapes ===")
_print_shape("time_conv", capture["temporal.time_conv"])
_print_shape("cat_conv (fused)", capture["temporal.cat_conv"])
print(f"\n  bev_fused: {tuple(outputs['bev_fused'].shape)}")

## Chunk 4 - Detection outputs and decoded 3D boxes

### Goal
Map BEV head outputs to query-style tensors and decoded boxes.

### Paper concept/equation
A dense BEV head predicts class logits and box fields per cell; top-k cells become query outputs.

### Explicit equations
`(E4.1)` Dense heads:

$$
H_{cls}, H_{reg} = f_{det}(\tilde{B})
$$

`(E4.2)` Metric conversion:

$$
x = \left(\sigma(\Delta x) + u\right)\cdot s_x + x_{min},\quad
y = \left(\sigma(\Delta y) + v\right)\cdot s_y + y_{min}
$$

`(E4.3)` Top-k decode:

$$
\hat{Y} = \mathrm{TopK}\left(\sigma(H_{cls}), H_{reg}\right)
$$

### Symbol table (E4.*)
- `(u, v)`: BEV grid coordinates
- `s_x, s_y`: metric scale from `pc_range`

### Code mapping
- `FBBEVDetectionHeadLite._decode_reg_map` in `pytorch_implementation/fbbev/detection_head.py`
- `FBBEVDetectionHeadLite._select_topk_queries` in `pytorch_implementation/fbbev/detection_head.py`
- `FBBEVBoxCoderLite.decode` in `pytorch_implementation/fbbev/postprocess.py`

### Tensor shape notes
- `all_cls_scores`: `[1, B, Q, num_classes]`
- `all_bbox_preds`: `[1, B, Q, 9]`

### One sanity check
Tests assert class/box output dimensions and finite values for all captures and outputs.

---

In [ ]:
# Chunk 4: Detection outputs and decoded 3D boxes
# Source: pytorch_implementation/fbbev/detection_head.py
# Source: pytorch_implementation/fbbev/postprocess.py

capture, handles = {}, []
_register_hook(model.detection_head.shared_conv, "head.shared", capture, handles)
_register_hook(model.detection_head.heatmap_head, "head.heatmap", capture, handles)
_register_hook(model.detection_head.reg_head, "head.reg", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

print("=== Detection head shapes ===")
_print_shape("shared_conv", capture["head.shared"])
_print_shape("heatmap_head", capture["head.heatmap"])
_print_shape("reg_head", capture["head.reg"])

print(f"\n=== Final outputs ===")
print(f"  all_cls_scores: {tuple(outputs['all_cls_scores'].shape)}")
print(f"  all_bbox_preds: {tuple(outputs['all_bbox_preds'].shape)}")

with torch.no_grad():
    decoded_out = model(img, img_metas, decode=True)
decoded = decoded_out["decoded"]
print(f"\n=== Decoded (sample 0) ===")
for key, val in decoded[0].items():
    if torch.is_tensor(val):
        print(f"  {key}: shape={tuple(val.shape)}, dtype={val.dtype}")

## 3) Dataflow diagram

```mermaid
flowchart LR
    imgInput["MultiCameraImage BxNcamx3xHxW"] --> backboneNeck[BackboneNeck]
    backboneNeck --> cameraFeat["camera_feat BxNcamxCxHfxWf"]
    cameraFeat --> depthNet[FBBEVDepthNetLite]
    depthNet --> context["context BxNcamxCxHfxWf"]
    depthNet --> depth["depth BxNcamxDxHfxWf"]
    context --> fwdProj[ForwardProjectionLite]
    depth --> fwdProj
    fwdProj --> bevVolume["bev_volume BxCxHbevxWbevxZbev"]
    bevVolume --> bwdProj[BackwardProjectionLite]
    cameraFeat --> bwdProj
    depth --> bwdProj
    bwdProj --> bevRefined["bev_refined BxCxHbevxWbev"]
    bevRefined --> temporal[TemporalFusionLite]
    imgMetas["img_metas ego_rt"] --> temporal
    temporal --> bevFused["bev_fused BxCxHbevxWbev"]
    bevFused --> bevEncoder[BEV Encoder convs]
    bevEncoder --> detHead[FBBEVDetectionHeadLite]
    detHead --> clsScores["all_cls_scores 1xBxQxCcls"]
    detHead --> bboxPreds["all_bbox_preds 1xBxQx9"]
    clsScores --> decode[FBBEVBoxCoderLite]
    bboxPreds --> decode
```

## 4) One end-to-end tensor trace

1. Start with `img [2, 6, 3, 96, 160]`.
2. Backbone+FPN returns camera features `[2, 6, 128, 6, 10]`.
3. DepthNet produces:
   - `context [2, 6, 128, 6, 10]` (projected camera features)
   - `depth [2, 6, 6, 6, 10]` (softmax depth distribution, 6 bins).
4. Forward projection lifts and aggregates:
   - `bev_volume [2, 128, 20, 20, 3]`.
5. Collapse Z dimension: `bev_2d [2, 128, 20, 20]`.
6. Backward projection refines with depth-aware attention:
   - `depth_weight [2, 1, 20, 20]`
   - `bev_refined [2, 128, 20, 20]`.
7. Temporal fusion aligns history with ego-motion and fuses:
   - `bev_fused [2, 128, 20, 20]`.
8. BEV encoder conv layers refine: `[2, 128, 20, 20]`.
9. Detection head produces dense cls/reg maps, then selects top-k queries:
   - `all_cls_scores [1, 2, 48, 10]`
   - `all_bbox_preds [1, 2, 48, 9]`.
10. Box coder decodes metric boxes/scores/labels.

## 5) Study drills (self-check questions)

1. Why does FB-BEV combine forward and backward projection instead of using only one?
2. What concrete tensors correspond to paper symbols `F`, `D`, `B`, and `B'`?
3. How does `ForwardProjectionLite` aggregate across cameras — averaging or concatenation?
4. What is the role of `depth_weight` in backward projection, and how is it computed?
5. Why does temporal fusion need `curr_to_prev_ego_rt` — what would happen without ego-motion alignment?
6. How does the detection head go from a dense BEV map to a query-style `[B, Q, ...]` output?
7. What is the purpose of `_meshgrid` in the detection head?
8. Why does the model use a Z dimension (`Zbev=3`) in the BEV volume that is later collapsed?
9. How does `start_of_sequence` affect temporal fusion behavior?
10. What would change if you doubled `depth_bins` — which tensors grow and which stay the same?

## 6) Practical reading order for this note

1. Read Sections 1 and 2 once.
2. Walk through Chunk 1 (depth net and forward projection) — understand the lift-splat path.
3. Study Chunk 2 (backward projection with depth-aware attention).
4. Study Chunk 3 (temporal fusion and ego-alignment).
5. Study Chunk 4 (detection head and top-k decode).
6. Re-read Chunk 0 (end-to-end) to tie the pipeline together.
7. Re-run the tensor trace in Section 4 while stepping through code.
8. Answer study drills without looking at code, then verify.

## 7) Known implementation simplifications in this repo

- Forward projection uses simple bilinear interpolation to scatter into BEV volume instead of pillar-based scatter.
- Temporal fusion queue depth is limited (typically 1 history frame) in the debug config.
- Backward projection uses a lightweight attention mechanism instead of full transformer cross-attention.
- Detection head uses dense conv prediction + top-k rather than learned query attention.

These simplifications keep the FB-BEV concept flow explicit for study.

In [ ]:
# Final finiteness validation
capture, handles = {}, []
_register_hook(model.backbone_neck.backbone.stem, "backbone.stem", capture, handles)
for idx, stage in enumerate(model.backbone_neck.backbone.stages):
    _register_hook(stage, f"backbone.stage{idx}", capture, handles)
with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()
_check_finite(capture, outputs)